# Multilingual Document OCR + MRZ Parser
### Domino Data Lab · Arabic / French / English · Fully Offline

---

## Bugs fixed in this version

| # | Error | Cause | Fix |
|---|-------|-------|-----|
| 1 | `AssertionError: param lang must in dict_keys(…), but got ar,fr,en` | PaddleOCR only accepts **one** language code per instance | Two engines: `lang='ar'` + `lang='fr'` |
| 2 | `gaierror / ConnectionError` on `paddleocr.bj.bcebos.com` | Domino has no internet — PaddleOCR tried to auto-download models | Pass `det_model_dir` + `rec_model_dir` so it reads from **local disk only** |

## Before running this notebook
Read **INSTRUCTIONS.txt** and complete Steps 1-3 first:
1. Download models on your **laptop terminal**
2. Upload models + images to **Domino** via the browser
3. Install packages in the **Domino terminal**

## Expected folder layout on Domino
```
/mnt/data/ocr_models/Multilingual_PP-OCRv3_det_infer/   ← detection model
/mnt/data/ocr_models/arabic_PP-OCRv3_rec_infer/         ← Arabic rec model
/mnt/data/ocr_models/latin_PP-OCRv3_rec_infer/          ← Latin rec model
/mnt/data/documents/                                    ← your images
/mnt/artifacts/                                         ← outputs (auto-created)
```

---
## Cell 1 — Imports

In [ ]:
# ── Standard library ─────────────────────────────────────────────────────────
import json
import logging
import os
import re
import time
from pathlib import Path
from typing import Any

# ── Third-party ───────────────────────────────────────────────────────────────
import cv2
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from tqdm.notebook import tqdm

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger(__name__)

print('✓ Cell 1 complete — all imports OK')

---
## Cell 2 — Configuration
**Only edit the paths in this cell if your folder names are different.**

In [ ]:
# =============================================================================
# PATHS — edit these if your folder names are different
# =============================================================================

# Where your document images are stored on Domino
INPUT_DIR    = Path('/mnt/data/documents')

# Where results will be saved (folders are created automatically)
OUTPUT_JSON  = Path('/mnt/artifacts/results.json')
OUTPUT_EXCEL = Path('/mnt/artifacts/report.xlsx')

# Root folder where you uploaded the model files
MODEL_BASE          = Path('/mnt/data/ocr_models')

# The three model sub-folders extracted from the .tar files
DET_MODEL_DIR       = MODEL_BASE / 'Multilingual_PP-OCRv3_det_infer'
REC_MODEL_DIR_AR    = MODEL_BASE / 'arabic_PP-OCRv3_rec_infer'
REC_MODEL_DIR_LATIN = MODEL_BASE / 'latin_PP-OCRv3_rec_infer'

# =============================================================================
# PERFORMANCE — adjust if needed
# =============================================================================

# Resize images larger than this before OCR (speeds up CPU inference)
MAX_DIMENSION = 1280

# Number of CPU threads for PaddleOCR inference
CPU_THREADS = os.cpu_count() or 4

# =============================================================================
# CONSTANTS — do not edit
# =============================================================================

SUPPORTED_EXT    = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif', '.webp'}
MRZ_LINE_PATTERN = re.compile(r'^[A-Z0-9<]{30,}$')

# =============================================================================
# VALIDATION — check model folders exist before doing anything else
# =============================================================================

missing = [
    d for d in [DET_MODEL_DIR, REC_MODEL_DIR_AR, REC_MODEL_DIR_LATIN]
    if not d.exists()
]

if missing:
    for m in missing:
        print(f'  ✗ NOT FOUND: {m}')
    raise FileNotFoundError(
        'Model directories are missing. '
        'Complete Steps 1 and 2 in INSTRUCTIONS.txt first.'
    )

print('✓ Cell 2 complete — configuration OK')
print(f'  Det model     : {DET_MODEL_DIR}')
print(f'  Arabic model  : {REC_MODEL_DIR_AR}')
print(f'  Latin model   : {REC_MODEL_DIR_LATIN}')
print(f'  Input images  : {INPUT_DIR}')
print(f'  CPU threads   : {CPU_THREADS}')
print(f'  Max image dim : {MAX_DIMENSION} px')

---
## Cell 3 — Image Preprocessing Helper

In [ ]:
def load_and_resize(
    image_path: str | Path,
    max_dim: int = MAX_DIMENSION,
) -> np.ndarray:
    """
    Load an image from disk and downscale if the longest side exceeds max_dim.

    Why resize?
        Reduces inference time on large scans without losing text readability.
        cv2.INTER_AREA gives the best quality when downscaling.

    Parameters
    ----------
    image_path : path to the image file
    max_dim    : maximum side length in pixels (default 1280)

    Returns
    -------
    BGR numpy array ready for PaddleOCR
    """
    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(f'Cannot read image: {image_path}')

    h, w  = img.shape[:2]
    scale = min(max_dim / max(h, w), 1.0)   # never upscale

    if scale < 1.0:
        new_w = int(w * scale)
        new_h = int(h * scale)
        img   = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
        log.debug('Resized %s  %dx%d → %dx%d',
                  Path(image_path).name, w, h, new_w, new_h)

    return img


print('✓ Cell 3 complete — load_and_resize defined')

---
## Cell 4 — Build the Two OCR Engines (offline)

### Why two engines?
PaddleOCR only accepts **one language code** per instance.  
Passing `lang='ar,fr,en'` raises `AssertionError`.  
We create **two engines** and merge their outputs:

| Engine | `rec_model_dir` | Recognises |
|--------|----------------|------------|
| `ocr_ar` | `arabic_PP-OCRv3_rec_infer` | Arabic script |
| `ocr_latin` | `latin_PP-OCRv3_rec_infer` | French + English |

### Why pass `det_model_dir` and `rec_model_dir`?
Without these, PaddleOCR tries to **download** models from `paddleocr.bj.bcebos.com`  
which fails on Domino (`gaierror: Name or service not known`).  
By pointing to the local folder, **zero network calls** are made.

> ⏱ This cell takes 10–30 seconds — normal, no download is happening.

In [ ]:
from paddleocr import PaddleOCR


def _make_engine(rec_model_dir: Path) -> PaddleOCR:
    """
    Create one PaddleOCR engine that reads ALL models from local disk.

    Parameters
    ----------
    rec_model_dir : Path
        Path to the language-specific recognition model folder.
        The detection model (DET_MODEL_DIR) is shared between both engines.

    Returns
    -------
    PaddleOCR instance (no network calls made)
    """
    return PaddleOCR(
        # ── FIX for ConnectionError: point to local model folders ─────────
        det_model_dir=str(DET_MODEL_DIR),   # shared detection model
        rec_model_dir=str(rec_model_dir),   # language-specific rec model
        # ── Inference settings ────────────────────────────────────────────
        use_angle_cls=False,    # documents assumed upright — saves time
        use_gpu=False,          # CPU-only
        enable_hpi=True,        # ONNX Runtime High-Performance Inference
        cpu_threads=CPU_THREADS,
        show_log=False,
    )


# ── Arabic engine ─────────────────────────────────────────────────────────────
print('Initialising Arabic engine  (reads from local disk, no download) …')
ocr_ar = _make_engine(REC_MODEL_DIR_AR)
print('  ✓ Arabic engine ready')

# ── Latin engine (French + English) ──────────────────────────────────────────
print('Initialising Latin engine   (reads from local disk, no download) …')
ocr_latin = _make_engine(REC_MODEL_DIR_LATIN)
print('  ✓ Latin engine ready  (French + English)')

print()
print('✓ Cell 4 complete — both OCR engines ready, fully offline')

---
## Cell 5 — MRZ Detection and Parsing

In [ ]:
def detect_mrz_lines(text_lines: list[str]) -> list[str]:
    """
    Scan OCR text lines for MRZ-like content using a regex heuristic.

    ICAO MRZ lines contain only uppercase letters A-Z, digits 0-9, and
    the '<' filler character, with a minimum length of 30 characters.
    (TD1=30 chars, TD2=36, TD3/passport=44)

    OCR sometimes inserts spaces inside MRZ lines — these are removed
    before matching.

    Parameters
    ----------
    text_lines : list of OCR text lines from PaddleOCR

    Returns
    -------
    list of cleaned MRZ line strings (spaces stripped)
    """
    mrz_lines: list[str] = []
    for line in text_lines:
        cleaned = line.strip().replace(' ', '')
        if MRZ_LINE_PATTERN.match(cleaned):
            mrz_lines.append(cleaned)
    return mrz_lines


def parse_mrz_with_omnimrz(mrz_lines: list[str]) -> dict[str, Any]:
    """
    Parse and validate MRZ lines using OmniMRZ (ICAO 9303 standard).

    Parameters
    ----------
    mrz_lines : MRZ lines from detect_mrz_lines()

    Returns
    -------
    dict containing:
        raw_mrz        — original line list
        parsed_fields  — structured ICAO fields (name, DOB, expiry, etc.)
        validation     — checksum results per field + overall_valid
        error          — only present if something went wrong
    """
    try:
        from omnimrz import MRZ
    except ImportError:
        log.warning('omnimrz not installed — MRZ will not be parsed structurally.')
        return {'error': 'omnimrz not installed', 'raw_mrz': mrz_lines}

    mrz_string = '\n'.join(mrz_lines)
    try:
        mrz_obj = MRZ(mrz_string)

        # ── Extract all ICAO 9303 fields ─────────────────────────────────
        fields: dict[str, Any] = {
            'document_type':    getattr(mrz_obj, 'document_type',    None),
            'issuing_country':  getattr(mrz_obj, 'issuing_country',  None),
            'document_number':  getattr(mrz_obj, 'document_number',  None),
            'surname':          getattr(mrz_obj, 'surname',          None),
            'given_names':      getattr(mrz_obj, 'given_names',      None),
            'nationality':      getattr(mrz_obj, 'nationality',      None),
            'birth_date':       getattr(mrz_obj, 'birth_date',       None),
            'sex':              getattr(mrz_obj, 'sex',              None),
            'expiry_date':      getattr(mrz_obj, 'expiry_date',      None),
            'optional_data_1':  getattr(mrz_obj, 'optional_data_1', None),
            'optional_data_2':  getattr(mrz_obj, 'optional_data_2', None),
        }

        # ── Checksum / validation flags ───────────────────────────────────
        validation: dict[str, Any] = {}
        for attr in dir(mrz_obj):
            if 'valid' in attr.lower() or 'check' in attr.lower():
                try:
                    validation[attr] = getattr(mrz_obj, attr)
                except Exception:
                    pass

        # Overall validity
        overall = getattr(mrz_obj, 'valid', None)
        if overall is None:
            overall = all(v for v in validation.values() if isinstance(v, bool))
        validation['overall_valid'] = overall

        return {
            'raw_mrz':       mrz_lines,
            'parsed_fields': fields,
            'validation':    validation,
        }

    except Exception as exc:
        log.warning('OmniMRZ parsing failed: %s', exc)
        return {'raw_mrz': mrz_lines, 'error': str(exc)}


print('✓ Cell 5 complete — MRZ helpers defined')

---
## Cell 6 — Single-Document Processor

In [ ]:
def _extract_lines(engine: Any, img: np.ndarray) -> list[str]:
    """
    Run one PaddleOCR engine on img and return a flat list of text strings.

    PaddleOCR returns:
        [ [ [bounding_box, (text, confidence)], ... ] ]
    This function unpacks that structure into a plain list of strings.
    """
    lines: list[str] = []
    raw = engine.ocr(img, cls=False)
    if raw and raw[0]:
        for item in raw[0]:
            if item and len(item) >= 2:
                txt = (
                    item[1][0]
                    if isinstance(item[1], (list, tuple))
                    else str(item[1])
                )
                lines.append(txt)
    return lines


def process_document(
    image_path:   str | Path,
    engine_ar:    Any,
    engine_latin: Any,
    max_dim:      int = MAX_DIMENSION,
) -> dict[str, Any]:
    """
    Run OCR on one document image and optionally parse MRZ.

    How it works:
      1. Load and resize the image.
      2. Run engine_ar   (Arabic)         → lines_ar
      3. Run engine_latin (French/English) → lines_latin
      4. Merge both lists, removing exact duplicates.
      5. Scan merged lines for MRZ pattern.
      6. If >= 2 MRZ lines found → parse with OmniMRZ.

    Parameters
    ----------
    image_path   : path to the document image
    engine_ar    : Arabic PaddleOCR engine
    engine_latin : Latin PaddleOCR engine (French + English)
    max_dim      : resize threshold in pixels

    Returns
    -------
    dict with keys:
        file_name   — filename of the image
        full_text   — all recognised text, newline-separated
        has_mrz     — True if MRZ was detected
        mrz_data    — parsed MRZ dict or None
        error       — error message or None
        elapsed_sec — processing time in seconds
    """
    image_path = Path(image_path)
    result: dict[str, Any] = {
        'file_name':   image_path.name,
        'full_text':   '',
        'has_mrz':     False,
        'mrz_data':    None,
        'error':       None,
        'elapsed_sec': 0.0,
    }
    t0 = time.perf_counter()

    try:
        # Step 1 — load and resize
        img = load_and_resize(image_path, max_dim)

        # Step 2+3 — run both engines
        lines_ar    = _extract_lines(engine_ar,    img)
        lines_latin = _extract_lines(engine_latin, img)

        # Step 4 — merge, Arabic first, remove duplicates
        seen: set[str]        = set()
        text_lines: list[str] = []
        for line in lines_ar + lines_latin:
            if line not in seen:
                seen.add(line)
                text_lines.append(line)

        result['full_text'] = '\n'.join(text_lines)
        log.debug('%s  ar=%d  latin=%d  merged=%d',
                  image_path.name, len(lines_ar), len(lines_latin), len(text_lines))

        # Step 5+6 — MRZ detection and parsing
        mrz_lines = detect_mrz_lines(text_lines)
        if len(mrz_lines) >= 2:
            result['has_mrz']  = True
            result['mrz_data'] = parse_mrz_with_omnimrz(mrz_lines)
            log.info('  ✓ MRZ detected (%d lines)', len(mrz_lines))
        else:
            log.info('  — No MRZ found')

    except Exception as exc:
        log.error('Error processing %s: %s', image_path.name, exc)
        result['error'] = str(exc)

    result['elapsed_sec'] = round(time.perf_counter() - t0, 3)
    return result


print('✓ Cell 6 complete — process_document defined')

---
## Cell 7 — Test with One Sample Image
Change `SAMPLE_IMAGE` to point to one of your document images.  
This is a quick sanity check before running the full batch.

In [ ]:
# ── Edit this path to point to one of your images ────────────────────────────
SAMPLE_IMAGE = Path('/mnt/data/documents/sample.jpg')   # ← change me

if SAMPLE_IMAGE.exists():
    print(f'Processing test image: {SAMPLE_IMAGE.name} …')
    test_result = process_document(SAMPLE_IMAGE, ocr_ar, ocr_latin)

    print(f'\n--- RESULT ---')
    print(f'File       : {test_result["file_name"]}')
    print(f'Has MRZ    : {test_result["has_mrz"]}')
    print(f'Time       : {test_result["elapsed_sec"]}s')
    print(f'Error      : {test_result["error"]}')
    print(f'\n--- FULL TEXT ---')
    print(test_result['full_text'])

    if test_result['has_mrz']:
        print(f'\n--- MRZ DATA ---')
        print(json.dumps(test_result['mrz_data'], ensure_ascii=False, indent=2, default=str))

    print('\n✓ Cell 7 complete — sample image processed')
else:
    print(f'[SKIP] {SAMPLE_IMAGE} not found.')
    print('Edit SAMPLE_IMAGE above to point to a real image, or go straight to Cell 8.')

---
## Cell 8 — Batch Processing
Processes **every image** found under `INPUT_DIR`.  
The progress bar shows one tick per document.

In [ ]:
def collect_images(input_path: Path) -> list[Path]:
    """
    Return a sorted list of image paths from a file or directory.
    If a directory is given, it is searched recursively.
    """
    if input_path.is_file():
        return [input_path]
    images = sorted(
        p for p in input_path.rglob('*')
        if p.suffix.lower() in SUPPORTED_EXT
    )
    if not images:
        log.warning('No images found in %s', input_path)
    return images


# ── Discover all images ───────────────────────────────────────────────────────
images = collect_images(INPUT_DIR)
print(f'Found {len(images)} image(s) in {INPUT_DIR}')

# ── Process each image ────────────────────────────────────────────────────────
all_results: list[dict[str, Any]] = []

for img_path in tqdm(images, desc='OCR + MRZ', unit='doc'):
    log.info('Processing: %s', img_path.name)
    r = process_document(img_path, ocr_ar, ocr_latin)
    all_results.append(r)
    log.info('  Done in %.2fs  |  has_mrz=%s  |  error=%s',
             r['elapsed_sec'], r['has_mrz'], r['error'])

total_time = sum(r['elapsed_sec'] for r in all_results)
print(f'\n✓ Cell 8 complete — {len(all_results)} document(s) processed in {total_time:.1f}s')

---
## Cell 9 — Save Results as JSON

In [ ]:
def save_json(results: list[dict[str, Any]], output_path: Path) -> None:
    """
    Write results to a pretty-printed UTF-8 JSON file.
    Parent directories are created automatically.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as fh:
        json.dump(results, fh, ensure_ascii=False, indent=2, default=str)
    size_kb = output_path.stat().st_size / 1024
    log.info('JSON saved → %s  (%.1f KB)', output_path, size_kb)


save_json(all_results, OUTPUT_JSON)
print(f'✓ Cell 9 complete — JSON saved to {OUTPUT_JSON}')

---
## Cell 10 — Save Results as Excel

Creates a 3-sheet workbook:

| Sheet | What it contains |
|-------|------------------|
| **Summary** | One row per document — filename, has_mrz, valid, time, error |
| **MRZ Details** | All parsed ICAO 9303 fields for documents with MRZ |
| **Full Text** | Complete OCR text for every document |

In [ ]:
def _apply_header_style(ws) -> None:
    """Bold font + light-blue fill on the header row."""
    fill = PatternFill(fill_type='solid', fgColor='BDD7EE')
    for cell in ws[1]:
        cell.font      = Font(bold=True)
        cell.fill      = fill
        cell.alignment = Alignment(horizontal='center', wrap_text=True)


def _autofit_columns(ws, max_width: int = 60) -> None:
    """Set column widths based on content (approximate)."""
    for col in ws.columns:
        letter = get_column_letter(col[0].column)
        best   = max((len(str(c.value or '')) for c in col), default=10)
        ws.column_dimensions[letter].width = min(best + 4, max_width)


def build_excel(results: list[dict[str, Any]], output_path: Path) -> None:
    """Build the 3-sheet formatted Excel report."""
    output_path.parent.mkdir(parents=True, exist_ok=True)

    summary_rows: list[dict] = []
    mrz_rows:     list[dict] = []
    text_rows:    list[dict] = []

    for r in results:
        mrz_data   = r.get('mrz_data') or {}
        validation = mrz_data.get('validation', {}) if r.get('has_mrz') else {}
        overall    = validation.get('overall_valid')

        # Sheet 1: Summary
        summary_rows.append({
            'file_name':         r.get('file_name'),
            'has_mrz':           r.get('has_mrz'),
            'overall_mrz_valid': overall,
            'elapsed_sec':       r.get('elapsed_sec'),
            'error':             r.get('error'),
        })

        # Sheet 3: Full Text
        text_rows.append({
            'file_name': r.get('file_name'),
            'full_text': r.get('full_text'),
        })

        # Sheet 2: MRZ Details (only for documents with MRZ)
        if r.get('has_mrz') and mrz_data:
            pf = mrz_data.get('parsed_fields', {}) or {}
            mrz_rows.append({
                'file_name':       r.get('file_name'),
                'raw_mrz':         ' | '.join(mrz_data.get('raw_mrz', [])),
                'document_type':   pf.get('document_type'),
                'issuing_country': pf.get('issuing_country'),
                'document_number': pf.get('document_number'),
                'surname':         pf.get('surname'),
                'given_names':     pf.get('given_names'),
                'nationality':     pf.get('nationality'),
                'birth_date':      pf.get('birth_date'),
                'sex':             pf.get('sex'),
                'expiry_date':     pf.get('expiry_date'),
                'optional_data_1': pf.get('optional_data_1'),
                'optional_data_2': pf.get('optional_data_2'),
                'overall_valid':   validation.get('overall_valid'),
                **{
                    f'valid_{k}': v
                    for k, v in validation.items()
                    if k != 'overall_valid'
                },
            })

    df_summary = pd.DataFrame(summary_rows)
    df_mrz     = (
        pd.DataFrame(mrz_rows) if mrz_rows
        else pd.DataFrame(columns=['file_name'])
    )
    df_text = pd.DataFrame(text_rows)

    with pd.ExcelWriter(str(output_path), engine='openpyxl') as writer:
        df_summary.to_excel(writer, sheet_name='Summary',     index=False)
        df_mrz.to_excel(    writer, sheet_name='MRZ Details', index=False)
        df_text.to_excel(   writer, sheet_name='Full Text',   index=False)

    wb = load_workbook(str(output_path))
    for ws in wb.worksheets:
        _apply_header_style(ws)
        _autofit_columns(ws)
    wb.save(str(output_path))

    size_kb = output_path.stat().st_size / 1024
    log.info('Excel saved → %s  (%.1f KB)', output_path, size_kb)


build_excel(all_results, OUTPUT_EXCEL)
print(f'✓ Cell 10 complete — Excel saved to {OUTPUT_EXCEL}')

---
## Cell 11 — Results Summary

In [ ]:
df_summary  = pd.read_excel(str(OUTPUT_EXCEL), sheet_name='Summary')
total_time  = sum(r['elapsed_sec'] for r in all_results)
with_mrz    = int(df_summary['has_mrz'].sum())
mrz_valid   = int(df_summary['overall_mrz_valid'].sum())
with_errors = int(df_summary['error'].notna().sum())

print('=' * 50)
print('  FINAL RESULTS SUMMARY')
print('=' * 50)
print(f'  Total documents processed : {len(df_summary)}')
print(f'  Documents with MRZ        : {with_mrz}')
print(f'  MRZ passed validation     : {mrz_valid}')
print(f'  Processing errors         : {with_errors}')
print(f'  Total processing time     : {total_time:.1f}s')
print(f'  Mean time per document    : {total_time/max(len(all_results),1):.2f}s')
print('=' * 50)
print(f'  JSON  saved → {OUTPUT_JSON}')
print(f'  Excel saved → {OUTPUT_EXCEL}')
print('=' * 50)
print()
print('Download your files from:  Domino → Artifacts')

# Show the table
df_summary